# Pattern 07 — Cost attribution

What this shows: `CostCalculator` turns model + token usage into a deterministic
per-decision cost using real provider pricing. Aggregated across decisions, this
is the ground truth for team-level AI spend attribution.
When to reach for it: any time you need to charge-back AI spend to teams,
models, or decisions — or forecast monthly cost given current traffic.
See also: patterns/01_decision_capture.py (decisions are what you attribute cost to)

In [ ]:
from briefcase.cost import CostCalculator

calc = CostCalculator()

## Per-decision cost for one model

In [ ]:
# estimate_cost takes model + input/output tokens and returns the cost in
# USD using real provider pricing compiled into the SDK.
c = calc.estimate_cost("gpt-4o", input_tokens=1200, output_tokens=350)
print(f"gpt-4o 1200 in / 350 out: ${c.total_cost:.6f}")
print(f"  input portion:  ${c.input_cost:.6f}")
print(f"  output portion: ${c.output_cost:.6f}")

## Cross-model comparison

In [ ]:
# estimate_cost returns a CostEstimate with total_cost, input_cost, etc.
# Running it for multiple models supports routing decisions such as
# "cheapest model that meets the quality bar."
for model in ["gpt-4o", "claude-3-5-sonnet", "gpt-4o-mini"]:
    try:
        est = calc.estimate_cost(model, 1200, 350)
        print(f"  {model:<25} ${est.total_cost:.6f}")
    except Exception as e:
        print(f"  {model:<25} (not registered: {type(e).__name__})")

## Monthly projection

In [ ]:
# Scale a single-decision cost by expected traffic. Use this for budget
# planning and proactive alerts before month-end overruns.
per_day = 50_000
per_call = calc.estimate_cost("gpt-4o-mini", 800, 200)
monthly = per_call.total_cost * per_day * 30
print(f"\nProjection for 50k gpt-4o-mini calls/day: ${monthly:,.2f}/month")

## Pricing tiers (rate cards)

`estimate_cost` takes a keyword-only `rate_card` to re-price under a `platform x tier x modifier`
scheme; the `batch` tier is half price. `CostEstimate.cache_cost` itemizes prompt-cache usage
(0 unless cache tokens are supplied). See patterns 12–14 for the full rate-card grammar,
prompt-cache cost, and multi-cloud pricing.

In [ ]:
std = calc.estimate_cost("gpt-4o", 1200, 350)
batch = calc.estimate_cost("gpt-4o", 1200, 350, rate_card="batch")
print(f"gpt-4o standard ${std.total_cost:.6f} vs batch ${batch.total_cost:.6f} "
      f"({batch.total_cost / std.total_cost:.2f}x); cache portion ${std.cache_cost:.6f}")